# **Statistical Analysis of Customer Spending**

Hypothesis tests on the segmented Mall Customers data (Module 4 hypotheses):

1. **Pairwise independent-samples t-tests:** do the customer segments differ in average spending score?
2. **Pearson correlation:** are annual income and age related to spending score?

> **Data:** reads `data/processed/mall_customers_clean.csv`, generated by `module-03-data-cleaning/data_cleaning.ipynb` (a copy is already included; see `data/README.md`).

In [1]:
from itertools import combinations
from pathlib import Path

import pandas as pd
from sklearn.cluster import KMeans
from scipy.stats import ttest_ind, pearsonr

## Load the cleaned dataset

In [2]:
DATA_PATH = Path('../data/processed/mall_customers_clean.csv')

df = pd.read_csv(DATA_PATH)
print(df.head())

   customer_id  gender  age  annual_income_k_usd  spending_score
0            1    Male   19                   15              39
1            2    Male   21                   15              81
2            3  Female   20                   16               6
3            4  Female   23                   16              77
4            5  Female   31                   17              40


## Customer segments (K-Means, k=5)

The cluster IDs below correspond to a K-Means run with `random_state=42` on income and spending score (same setup as Module 5). If your data differs, re-check the mapping against a scatter plot before running the tests.

In [3]:
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster'] = kmeans.fit_predict(df[['annual_income_k_usd', 'spending_score']])

cluster_names = {
    0: 'Mid Income / Mid Score',
    1: 'High Income / High Score',
    2: 'Low Income / High Score',
    3: 'High Income / Low Score',
    4: 'Low Income / Low Score'
}

df['cluster_name'] = df['cluster'].map(cluster_names)

df.head()

,customer_id,gender,age,annual_income_k_usd,spending_score,cluster,cluster_name
0,1,Male,19,15,39,4,Low Income / Low Score
1,2,Male,21,15,81,2,Low Income / High Score
2,3,Female,20,16,6,4,Low Income / Low Score
3,4,Female,23,16,77,2,Low Income / High Score
4,5,Female,31,17,40,4,Low Income / Low Score


## Analysis 1 - Spending score between segment pairs

**Question:** Which pairs of customer segments show a statistically significant difference in mean `spending_score`?

**Test:** Independent-samples t-test applied pairwise to every combination of segments.

In [4]:
segment_list = df['cluster_name'].unique()
results = []

for a, b in combinations(segment_list, 2):
    group1 = df[df['cluster_name'] == a]['spending_score']
    group2 = df[df['cluster_name'] == b]['spending_score']
    t_stat, p_val = ttest_ind(group1, group2)
    results.append({
        'comparison': f'{a} vs {b}',
        't_stat': t_stat,
        'p_value': p_val
    })

t_test_table = pd.DataFrame(results)
t_test_table

,comparison,t_stat,p_value
0,Low Income / Low Score vs Low Income / High Score,-16.530918,3.028276e-20
1,Low Income / Low Score vs Mid Income / Mid Score,-14.470663,1.820071e-26
2,Low Income / Low Score vs High Income / High S...,-21.464806,7.632948e-30
3,Low Income / Low Score vs High Income / Low Score,1.257280,2.138706e-01
4,Low Income / High Score vs Mid Income / Mid Score,16.482305,2.149569e-30
5,Low Income / High Score vs High Income / High ...,-1.059569,2.936595e-01
6,Low Income / High Score vs High Income / Low S...,22.504681,2.018553e-29
7,Mid Income / Mid Score vs High Income / High S...,-22.130658,8.159717e-44
8,Mid Income / Mid Score vs High Income / Low Score,20.772769,1.473581e-40
9,High Income / High Score vs High Income / Low ...,28.946019,2.137190e-41


## Analysis 2 - Factors related to spending score

**Question:** Is there a (linear) correlation between annual income and spending score, and between age and spending score?

**Test:** Pearson correlation.

In [5]:
income = df['annual_income_k_usd']
age = df['age']
spending_score = df['spending_score']

corr_income, p_income = pearsonr(income, spending_score)

corr_age, p_age = pearsonr(age, spending_score)

print("--- Correlation: Annual Income vs Spending Score ---")
print(f"Correlation coefficient: {corr_income:.4f}")
print(f"P-value: {p_income:.4f}\n")

print("--- Correlation: Age vs Spending Score ---")
print(f"Correlation coefficient: {corr_age:.4f}")
print(f"P-value: {p_age:.4f}")

--- Correlation: Annual Income vs Spending Score ---
Correlation coefficient: 0.0099
P-value: 0.8890

--- Correlation: Age vs Spending Score ---
Correlation coefficient: -0.3272
P-value: 0.0000
